In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import requests
import cv2
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision.models import resnet50, ResNet50_Weights
from sklearn.cluster import KMeans
from sklearn.model_selection import GroupShuffleSplit

# ======================================================
# 1. GEOGRAPHIC UTILITIES & DATA [cite: 22, 58]
# ======================================================

def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculates great-circle distance [cite: 63]"""
    R = 6371.0 # Earth's radius [cite: 65]
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

def latlon_to_xyz(lat, lon):
    lat, lon = np.radians(lat), np.radians(lon)
    return np.stack([np.cos(lat)*np.cos(lon), np.cos(lat)*np.sin(lon), np.sin(lat)], axis=1)

def generate_voronoi_regions(coords, k):
    """Discretized Global Inference via K-region discretization """
    xyz = latlon_to_xyz(coords[:,0], coords[:,1])
    kmeans = KMeans(n_clusters=min(k, len(coords)), random_state=42, n_init=10)
    labels = kmeans.fit_predict(xyz)
    centroids_xyz = kmeans.cluster_centers_
    lat = np.degrees(np.arcsin(centroids_xyz[:,2]))
    lon = np.degrees(np.arctan2(centroids_xyz[:,1], centroids_xyz[:,0]))
    return np.stack([lat, lon], axis=1), labels

# ======================================================
# 2. DATASET CLASS WITH AUGMENTATION [cite: 30, 31, 32]
# ======================================================

class MapillaryDataset(Dataset):
    def __init__(self, df, is_train=False):
        self.df = df.reset_index(drop=True)
        self.is_train = is_train
        normalize = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

        if self.is_train:
            self.transform = T.Compose([
                T.RandomResizedCrop(224, scale=(0.7, 1.0)), # Mitigate sequence bias [cite: 17]
                T.ColorJitter(0.3, 0.3, 0.3, 0.1),
                T.RandomHorizontalFlip(),
                T.ToTensor(),
                normalize,
                T.RandomErasing(p=0.3, scale=(0.02, 0.2)) # Force focus on high-level cues [cite: 17]
            ])
        else:
            self.transform = T.Compose([
                T.Resize((256, 256)),
                T.CenterCrop(224),
                T.ToTensor(),
                normalize
            ])

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['image_path']).convert('RGB')
        return (self.transform(img), 
                torch.tensor(row['country_idx']).long(), 
                torch.tensor(row['region_idx']).long(), 
                torch.tensor([row['lat'], row['lon']], dtype=torch.float32))

# ======================================================
# 3. HIERARCHICAL MODEL [cite: 19, 34, 37]
# ======================================================

class GeoGuessrBot(nn.Module):
    def __init__(self, num_countries, num_regions):
        super().__init__()
        # Transition from VGG to ResNet-50 
        backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        self.features = nn.Sequential(*list(backbone.children())[:-2])
        self.avgpool = backbone.avgpool
        dim = backbone.fc.in_features
        
        # 5.1 & 5.2 Hierarchical Heads [cite: 38, 50]
        self.country_head = nn.Linear(dim, num_countries)
        self.region_head = nn.Linear(dim, num_regions)
        self.gradients = None

    def activations_hook(self, grad): self.gradients = grad

    def forward(self, x):
        h = self.features(x) # Image feature vector h(x) [cite: 36]
        if x.requires_grad:
            h.register_hook(self.activations_hook)
        
        pooled = torch.flatten(self.avgpool(h), 1)
        # Linear mappings to logits: z = Wh(x) + b [cite: 40, 52]
        return self.country_head(pooled), self.region_head(pooled), h

    def get_activations_gradient(self): return self.gradients

# ======================================================
# 4. LOSS & INTERPRETABILITY [cite: 68, 69]
# ======================================================

class HierarchicalLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()
        
    def forward(self, c_out, c_target, r_out, r_target):
        return self.ce(c_out, c_target) + self.ce(r_out, r_target)

def generate_gradcam(model, img_tensor, pred_idx):
    """Identifies visual cues like signage and vegetation [cite: 21, 69, 71]"""
    model.zero_grad()
    c_logits, r_logits, features = model(img_tensor.unsqueeze(0))
    r_logits[0, pred_idx].backward()
    
    grads = model.get_activations_gradient()
    weights = torch.mean(grads, dim=[0, 2, 3])
    activations = features.detach()
    
    for i in range(activations.size(1)):
        activations[:, i, :, :] *= weights[i]
        
    heatmap = torch.mean(activations, dim=1).squeeze()
    heatmap = np.maximum(heatmap.cpu(), 0)
    return heatmap / (torch.max(heatmap) + 1e-10)

# ======================================================
# 5. EVALUATION [cite: 58, 60]
# ======================================================

def evaluate(model, loader, centroids, device):
    model.eval()
    errors = []
    with torch.no_grad():
        for imgs, _, _, true_coords in loader:
            _, r_logits, _ = model(imgs.to(device))
            pred_k = torch.argmax(r_logits, dim=1).cpu().numpy()
            for i, k in enumerate(pred_k):
                p_lat, p_lon = centroids[k] # Predicted coordinates [cite: 57]
                t_lat, t_lon = true_coords[i].numpy()
                errors.append(haversine_distance(t_lat, t_lon, p_lat, p_lon))
    return np.median(errors)

In [ ]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
import pandas as pd
import os

# Assuming the previously defined classes (MapillaryDataset, GeoGuessrBot, 
# HierarchicalLoss) and utilities (generate_voronoi_regions, evaluate) are imported.

def main():
    # ------------------------------------------------------
    # 1. CONFIGURATION & HYPERPARAMETERS
    # ------------------------------------------------------
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    NUM_REGIONS = 1000  # K-region discretization [cite: 20]
    BATCH_SIZE = 32
    LEARNING_RATE = 1e-4
    EPOCHS = 15
    SAVE_DIR = "checkpoints"
    os.makedirs(SAVE_DIR, exist_ok=True)

    print(f"Starting execution on {DEVICE}...")

    # ------------------------------------------------------
    # 2. DATA PREPARATION [cite: 22, 30]
    # ------------------------------------------------------
    # Load metadata containing [lat, lon, country_idx, image_path] [cite: 24, 29]
    if not os.path.exists("metadata.csv"):
        raise FileNotFoundError("Metadata file not found. Ensure Mapillary data is fetched.")
    
    metadata = pd.read_csv("metadata.csv")
    coords = metadata[['lat', 'lon']].values
    
    # Hierarchical approach: Discretize global coordinates [cite: 20]
    centroids, region_labels = generate_voronoi_regions(coords, NUM_REGIONS)
    metadata['region_idx'] = region_labels
    
    # Spatial Split: Ensure no sequence bias by grouping by region [cite: 17, 33]
    splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
    train_idx, val_idx = next(splitter.split(metadata, groups=metadata.region_idx))
    
    train_ds = MapillaryDataset(metadata.iloc[train_idx], is_train=True)
    val_ds = MapillaryDataset(metadata.iloc[val_idx], is_train=False)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

    # ------------------------------------------------------
    # 3. MODEL INITIALIZATION [cite: 19, 37]
    # ------------------------------------------------------
    num_countries = metadata['country_idx'].nunique()
    model = GeoGuessrBot(num_countries=num_countries, num_regions=NUM_REGIONS).to(DEVICE)
    criterion = HierarchicalLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    # ------------------------------------------------------
    # 4. TRAINING LOOP [cite: 12]
    # ------------------------------------------------------
    for epoch in range(EPOCHS):
        model.train()
        total_train_loss = 0
        
        for imgs, c_targets, r_targets, _ in train_loader:
            imgs, c_targets, r_targets = imgs.to(DEVICE), c_targets.to(DEVICE), r_targets.to(DEVICE)
            
            optimizer.zero_grad()
            c_logits, r_logits, _ = model(imgs) # Hierarchical heads [cite: 37]
            
            loss = criterion(c_logits, c_targets, r_logits, r_targets)
            loss.backward()
            optimizer.step()
            
            total_train_loss += loss.item()
            
        # ------------------------------------------------------
        # 5. EVALUATION [cite: 58, 59]
        # ------------------------------------------------------
        median_km_error = evaluate(model, val_loader, centroids, DEVICE)
        
        print(f"Epoch [{epoch+1}/{EPOCHS}] - "
              f"Loss: {total_train_loss/len(train_loader):.4f} - "
              f"Median Error: {median_km_error:.2f} km")
        
        # Save best model
        torch.save(model.state_dict(), f"{SAVE_DIR}/geoguessr_best.pth")

    # ------------------------------------------------------
    # 6. INTERPRETABILITY REPORT [cite: 21, 68]
    # ------------------------------------------------------
    print("Generating visual feature interpretation reports...")
    # Analyzing focus on road markings, vegetation, and architecture [cite: 70, 71, 72]
    batch_interpretability_report(model, val_loader, DEVICE)

if __name__ == "__main__":
    main()